# CIFAR-10 Single-Head ViT with Learned LayerNorm and Standard Softmax

This notebook keeps the same small CIFAR-10 ViT setup (single head, 3 Transformer blocks) and trains from scratch with the **learned dynamic LayerNorm replacement** but **standard PyTorch Softmax**.

### Learned LayerNorm replacement

For each token $x_t \in \mathbb{R}^D$:

$$a_t=\mathrm{MLP}_{D\rightarrow32\rightarrow1}(x_t)$$

$$y_t=a_tx_t+\beta.$$

The dynamic state is still one scalar slope per token. There is no explicit mean, variance, square root, or division in the learned normalization layer.

### Standard Softmax

Attention weights are computed with

```python
attn = torch.softmax(scores, dim=-1)
```

This notebook isolates the effect of the learned LayerNorm replacement while keeping attention normalization standard.


In [1]:
# ============================================================
# Imports and configuration
# ============================================================

import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

DATA_ROOT = "./data"
OUT_DIR = "./runs/cifar10_singlehead_vit_learned_norm_standard_softmax"

EPOCHS = 400
BATCH_SIZE = 256
NUM_WORKERS = 4
# LR = 3e-4
LR = 1e-3
WEIGHT_DECAY = 0.05
SEED = 42

IMAGE_SIZE = 32
PATCH_SIZE = 4
DIM = 128
DEPTH = 3
MLP_RATIO = 4.0
DROPOUT = 0.1
ATTN_DROPOUT = 0.0


In [2]:
# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


## 1. Learned LayerNorm replacement


In [3]:
# ============================================================
# Learned LayerNorm replacement
# ============================================================

class LearnedLayerNormReplacement(nn.Module):
    """
    Dynamic affine replacement for LayerNorm:

        [a_t, b_t] = MLP(x_t)   # [B, N, D] -> [B, N, 2]
        y_t = a_t * x_t + b_t + beta

    The same tiny MLP is shared across all tokens.
    """

    def __init__(self, dim):
        super().__init__()

        self.predictor_0 = nn.Linear(
            dim,
            32,
            bias=True,
        )

        self.predictor_relu = nn.ReLU()

        self.predictor_1 = nn.Linear(
            32,
            2,
            bias=True,
        )

        # # Static learned per-channel bias.
        # self.beta = nn.Parameter(
        #     torch.zeros(dim)
        # )

        nn.init.kaiming_uniform_(
            self.predictor_0.weight,
            nonlinearity='relu'
        )
        nn.init.zeros_(self.predictor_0.bias)

        # Keep initial a ~= 1 and b ~= 0.
        nn.init.normal_(
            self.predictor_1.weight,
            mean=0.0,
            std=1e-3
        )

        with torch.no_grad():
            self.predictor_1.bias[0] = 1.0   # a
            self.predictor_1.bias[1] = 0.0   # b

    def forward(self, x):
        state = self.predictor_0(x)           # [B, N, 32]
        state = self.predictor_relu(state)
        state = self.predictor_1(state)       # [B, N, 2]

        raw_a = state[..., 0:1]               # [B, N, 1]
        b = state[..., 1:2]                   # [B, N, 1]

        a = torch.clamp(
            raw_a,
            min=0.05,
            max=12.0,
        )

        # return a * x + b + self.beta
        return a * x + b


## 2. Standard Softmax


In [4]:
# ============================================================
# Standard Softmax
# ============================================================

# This ablation uses torch.softmax(scores, dim=-1) directly inside
# SingleHeadSelfAttention. No recurrent Softmax replacement is used.


## 3. Single-head attention

In [5]:
# ============================================================
# Single-head self-attention with standard Softmax
# ============================================================

class SingleHeadSelfAttention(nn.Module):

    def __init__(
        self,
        dim,
        attn_dropout=0.0,
        proj_dropout=0.0,
    ):
        super().__init__()

        self.dim = dim
        self.scale = dim ** -0.5

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

        self.attn_dropout = nn.Dropout(attn_dropout)
        self.proj_dropout = nn.Dropout(proj_dropout)

    def forward(self, x):
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        scores = torch.matmul(
            q,
            k.transpose(-2, -1),
        ) * self.scale

        # Standard Softmax ablation.
        attn = torch.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)

        out = torch.matmul(attn, v)
        out = self.out_proj(out)
        out = self.proj_dropout(out)

        return out


## 4. MLP and Transformer block

In [6]:
# ============================================================
# MLP
# ============================================================

class MLP(nn.Module):

    def __init__(self, dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


# ============================================================
# Transformer block
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(
        self,
        dim,
        num_tokens,
        mlp_ratio=4.0,
        dropout=0.0,
        attn_dropout=0.0,
    ):
        super().__init__()

        self.norm1 = LearnedLayerNormReplacement(dim)

        self.attn = SingleHeadSelfAttention(
            dim=dim,
            attn_dropout=attn_dropout,
            proj_dropout=dropout,
        )

        self.norm2 = LearnedLayerNormReplacement(dim)

        self.mlp = MLP(
            dim=dim,
            hidden_dim=int(dim * mlp_ratio),
            dropout=dropout,
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


## 5. CIFAR-10 ViT

In [7]:
# ============================================================
# CIFAR-10 single-head ViT
# ============================================================

class CIFAR10SingleHeadViT(nn.Module):

    def __init__(
        self,
        image_size=32,
        patch_size=4,
        in_channels=3,
        num_classes=10,
        dim=128,
        depth=3,
        mlp_ratio=4.0,
        dropout=0.1,
        attn_dropout=0.0,
    ):
        super().__init__()

        assert image_size % patch_size == 0

        self.image_size = image_size
        self.patch_size = patch_size
        self.dim = dim

        patches_per_side = image_size // patch_size
        self.num_patches = patches_per_side ** 2
        self.num_tokens = self.num_patches + 1

        patch_dim = in_channels * patch_size * patch_size

        self.patch_proj = nn.Linear(patch_dim, dim)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_tokens, dim)
        )
        self.pos_drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(
                dim=dim,
                num_tokens=self.num_tokens,
                mlp_ratio=mlp_ratio,
                dropout=dropout,
                attn_dropout=attn_dropout,
            )
            for _ in range(depth)
        ])

        # Replace final LayerNorm as well.
        self.final_norm = LearnedLayerNormReplacement(dim)
        self.head = nn.Linear(dim, num_classes)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # Initialize ordinary Linear layers as in the previous baseline,
        # while preserving the special initialization of the learned
        # LayerNorm scale predictors.
        for name, module in self.named_modules():
            if not isinstance(module, nn.Linear):
                continue

            if "scale_predictor" in name:
                continue

            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def patchify(self, x):
        p = self.patch_size

        x = x.unfold(2, p, p).unfold(3, p, p)
        x = x.permute(0, 2, 3, 1, 4, 5).contiguous()
        x = x.view(x.size(0), self.num_patches, -1)
        return x

    def forward(self, x):
        x = self.patchify(x)
        x = self.patch_proj(x)

        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1)

        x = x + self.pos_embed
        x = self.pos_drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        cls_out = x[:, 0]
        logits = self.head(cls_out)
        return logits


def build_model():
    return CIFAR10SingleHeadViT(
        image_size=IMAGE_SIZE,
        patch_size=PATCH_SIZE,
        num_classes=10,
        dim=DIM,
        depth=DEPTH,
        mlp_ratio=MLP_RATIO,
        dropout=DROPOUT,
        attn_dropout=ATTN_DROPOUT,
    )


## 6. CIFAR-10 data loaders

In [8]:
# ============================================================
# Dataset
# ============================================================

def make_loaders(
    data_root="./data",
    batch_size=256,
    num_workers=4,
    seed=42,
):

    train_tf = transforms.Compose([
        transforms.RandomCrop(
            32,
            padding=4,
        ),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(
                0.4914,
                0.4822,
                0.4465,
            ),
            std=(
                0.2470,
                0.2435,
                0.2616,
            ),
        ),
    ])

    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(
                0.4914,
                0.4822,
                0.4465,
            ),
            std=(
                0.2470,
                0.2435,
                0.2616,
            ),
        ),
    ])

    full_train_aug = datasets.CIFAR10(
        root=data_root,
        train=True,
        download=True,
        transform=train_tf,
    )

    full_train_eval = datasets.CIFAR10(
        root=data_root,
        train=True,
        download=False,
        transform=eval_tf,
    )

    test_set = datasets.CIFAR10(
        root=data_root,
        train=False,
        download=True,
        transform=eval_tf,
    )

    g = torch.Generator().manual_seed(seed)

    indices = torch.randperm(
        len(full_train_aug),
        generator=g,
    ).tolist()

    train_indices = indices[:45_000]
    val_indices = indices[45_000:]

    train_set = Subset(
        full_train_aug,
        train_indices,
    )

    # Same training samples, but augmentation disabled.
    train_eval_set = Subset(
        full_train_eval,
        train_indices,
    )

    val_set = Subset(
        full_train_eval,
        val_indices,
    )

    loader_kwargs = dict(
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
    )

    train_loader = DataLoader(
        train_set,
        shuffle=True,
        drop_last=True,
        **loader_kwargs,
    )

    train_eval_loader = DataLoader(
        train_eval_set,
        shuffle=False,
        drop_last=False,
        **loader_kwargs,
    )

    val_loader = DataLoader(
        val_set,
        shuffle=False,
        drop_last=False,
        **loader_kwargs,
    )

    test_loader = DataLoader(
        test_set,
        shuffle=False,
        drop_last=False,
        **loader_kwargs,
    )

    return (
        train_loader,
        train_eval_loader,
        val_loader,
        test_loader,
    )


train_loader, train_eval_loader, val_loader, test_loader = make_loaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seed=SEED,
)

print(
    len(train_loader.dataset),
    len(val_loader.dataset),
    len(test_loader.dataset),
)


45000 5000 10000


### Numerical stabilization used in this version

This ablation keeps **standard Softmax** and only replaces LayerNorm:

- learned LayerNorm: $y=a(x)x+\beta$
- attention normalization: standard `torch.softmax`

Two safeguards are retained for the learned LayerNorm path:

1. $a(x)$ is clipped to $[0.05,12]$ so the dynamic affine layer cannot instantly explode or invert sign.
2. Gradient norm is clipped to 1.0.

There is no LayerNorm teacher, auxiliary regression loss, explicit normalization loss, or quantization.


## 7. Training and evaluation

In [9]:
# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader,
    device,
):
    model.eval()

    criterion = nn.CrossEntropyLoss()

    total = 0
    correct = 0
    loss_sum = 0.0

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True,
        )

        labels = labels.to(
            device,
            non_blocking=True,
        )

        logits = model(images)
        loss = criterion(
            logits,
            labels,
        )

        total += labels.size(0)

        correct += (
            logits.argmax(dim=1)
            == labels
        ).sum().item()

        loss_sum += (
            loss.item()
            * labels.size(0)
        )

    return (
        loss_sum / total,
        correct / total,
    )


# ============================================================
# Training
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=100,
    lr=3e-4,
    weight_decay=0.05,
    out_dir="./runs/cifar10_singlehead_vit_learned_norm_standard_softmax",
):

    out_dir = Path(out_dir)
    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=epochs,
        )
    )

    best_val_acc = 0.0

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        total = 0
        correct = 0
        loss_sum = 0.0

        for images, labels in train_loader:

            images = images.to(
                device,
                non_blocking=True,
            )

            labels = labels.to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(images)

            loss = criterion(
                logits,
                labels,
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    "Non-finite loss detected. "
                    "The learned LayerNorm replacement is numerically unstable."
                )

            loss.backward()

            # Keep the dynamic slope predictor numerically stable
            # during early training.
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            total += labels.size(0)

            correct += (
                logits.argmax(dim=1)
                == labels
            ).sum().item()

            loss_sum += (
                loss.item()
                * labels.size(0)
            )

        scheduler.step()

        train_loss = loss_sum / total
        train_acc = correct / total

        val_loss, val_acc = evaluate(
            model,
            val_loader,
            device,
        )

        print(
            f"Epoch {epoch:03d}/{epochs} | "
            f"lr={scheduler.get_last_lr()[0]:.3e} | "
            f"train loss={train_loss:.4f} "
            f"acc={train_acc:.4f} | "
            f"val loss={val_loss:.4f} "
            f"acc={val_acc:.4f}"
        )

        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "val_acc": val_acc,
            },
            out_dir / "last.pt",
        )

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            torch.save(
                {
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "val_acc": val_acc,
                },
                out_dir / "best.pt",
            )

    print(
        f"Best validation accuracy: "
        f"{best_val_acc:.4f}"
    )


def load_checkpoint(
    model,
    ckpt_path,
    device,
):

    ckpt = torch.load(
        ckpt_path,
        map_location=device,
    )

    model.load_state_dict(
        ckpt["model"]
    )

    print(
        f"Loaded checkpoint: {ckpt_path} | "
        f"epoch={ckpt.get('epoch', '?')} | "
        f"val_acc={ckpt.get('val_acc', '?')}"
    )


## 8. Build and train from scratch

In [10]:
# ============================================================
# Build model
# ============================================================

model = build_model().to(device)

num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(model)
print(f"Parameters: {num_params:,}")


CIFAR10SingleHeadViT(
  (patch_proj): Linear(in_features=48, out_features=128, bias=True)
  (pos_drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-2): 3 x TransformerBlock(
      (norm1): LearnedLayerNormReplacement(
        (predictor_0): Linear(in_features=128, out_features=32, bias=True)
        (predictor_relu): ReLU()
        (predictor_1): Linear(in_features=32, out_features=2, bias=True)
      )
      (attn): SingleHeadSelfAttention(
        (q_proj): Linear(in_features=128, out_features=128, bias=True)
        (k_proj): Linear(in_features=128, out_features=128, bias=True)
        (v_proj): Linear(in_features=128, out_features=128, bias=True)
        (out_proj): Linear(in_features=128, out_features=128, bias=True)
        (attn_dropout): Dropout(p=0.0, inplace=False)
        (proj_dropout): Dropout(p=0.1, inplace=False)
      )
      (norm2): LearnedLayerNormReplacement(
        (predictor_0): Linear(in_features=128, out_features=32, bias=True)
        (predict

In [11]:
# ============================================================
# Train from scratch
# ============================================================

train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    out_dir=OUT_DIR,
)


Epoch 001/400 | lr=1.000e-03 | train loss=2.1288 acc=0.1809 | val loss=2.0266 acc=0.2092
Epoch 002/400 | lr=9.999e-04 | train loss=2.0240 acc=0.2192 | val loss=1.9769 acc=0.2232
Epoch 003/400 | lr=9.999e-04 | train loss=1.9736 acc=0.2445 | val loss=1.9673 acc=0.2440
Epoch 004/400 | lr=9.998e-04 | train loss=1.9122 acc=0.2746 | val loss=1.9369 acc=0.2864
Epoch 005/400 | lr=9.996e-04 | train loss=1.8520 acc=0.2980 | val loss=1.8497 acc=0.2982
Epoch 006/400 | lr=9.994e-04 | train loss=1.8201 acc=0.3092 | val loss=1.8144 acc=0.3206
Epoch 007/400 | lr=9.992e-04 | train loss=1.7921 acc=0.3224 | val loss=1.7842 acc=0.3264
Epoch 008/400 | lr=9.990e-04 | train loss=1.7730 acc=0.3293 | val loss=1.7470 acc=0.3492
Epoch 009/400 | lr=9.988e-04 | train loss=1.7505 acc=0.3396 | val loss=1.7445 acc=0.3420
Epoch 010/400 | lr=9.985e-04 | train loss=1.7352 acc=0.3456 | val loss=1.7133 acc=0.3434
Epoch 011/400 | lr=9.981e-04 | train loss=1.7139 acc=0.3513 | val loss=1.7126 acc=0.3404
Epoch 012/400 | lr=9.

In [12]:
# ============================================================
# Load best checkpoint and evaluate
# ============================================================

best_ckpt = Path(OUT_DIR) / "best.pt"
load_checkpoint(model, best_ckpt, device)

val_loss, val_acc = evaluate(model, val_loader, device)
test_loss, test_acc = evaluate(model, test_loader, device)

print(f"Validation: loss={val_loss:.4f}, acc={val_acc:.4f}")
print(f"Test:       loss={test_loss:.4f}, acc={test_acc:.4f}")


Loaded checkpoint: runs/cifar10_singlehead_vit_learned_norm_standard_softmax/best.pt | epoch=363 | val_acc=0.7888
Validation: loss=0.6400, acc=0.7888
Test:       loss=0.6450, acc=0.7899


## Notes

This notebook is the **learned LayerNorm + standard Softmax** ablation. It intentionally uses no LayerNorm teacher, auxiliary regression loss, explicit normalization loss, or quantization. The only learned replacement being tested is the dynamic affine LayerNorm mechanism.
